In [ ]:
import pandas as pd
from IPython.display import Image
from IPython.display import display as ipy_display

In [ ]:
import os

from ugbio_ppmseq.ppmSeq_utils import (
    STRAND_RATIO_LOWER_THRESH,
    STRAND_RATIO_UPPER_THRESH,
    PpmseqAdapterVersions,
    _assert_adapter_version_supported,
)

In [ ]:
IMAGE_WIDTH = 800

In [ ]:
# input parameters
adapter_version = None
statistics_h5 = None
strand_ratio_category_png = None
strand_ratio_category_concordance_png = None
sr_hist_png = None
sr_by_et_png = None
sr_lower = STRAND_RATIO_LOWER_THRESH
sr_upper = STRAND_RATIO_UPPER_THRESH
illustration_file = None
trimmer_failure_codes_csv = None
output_read_length_histogram_plot = None

In [ ]:
if statistics_h5 is None or adapter_version is None or strand_ratio_category_png is None:
    raise ValueError("Missing required input files")

In [ ]:
_assert_adapter_version_supported(adapter_version)

# Main statistics

In [ ]:
sorter_stats_shortlist = pd.read_hdf(statistics_h5, key="stats_shortlist")
if isinstance(sorter_stats_shortlist, pd.Series):
    sorter_stats_shortlist = sorter_stats_shortlist.to_frame()
ipy_display(sorter_stats_shortlist.style.format("{:.2f}"))
print("\n\n\n\n")

- MIXED_read_mean_coverage is the coverage of reads where the tag in both the start and the end of the read were detected as MIXED
- pct_MIXED_both_tags is the ratio of reads where both loops were detected as MIXED out of all the reads
- pct_MIXED_both_tags_where_endreached is the ratio of reads where both loops were detected as MIXED out of the reads where the read end was reached so that the end loop could be measured

# QC plots

In [ ]:
ipy_display(Image(strand_ratio_category_png, width=IMAGE_WIDTH))
caption = {
    PpmseqAdapterVersions.LEGACY_V5_START.value: """
This barplot shows the ratio of each category type in the data according to the spec in the top of the file.""",
    PpmseqAdapterVersions.LEGACY_V5_END.value: """
This barplot shows the ratio of each category type in the data according to the spec in the top of the file.
The end loop breakdown is only for the reads that reached the end loop.""",
    PpmseqAdapterVersions.LEGACY_V5.value: """
This barplot shows the ratio of each category type in the data according to the spec in the top of the file.
The categories are reported separately for the start- and end-loops.
The end loop breakdown is shown only for the reads that reached the end loop.""",
    PpmseqAdapterVersions.V1.value: """
This barplot shows the ratio of each category type in the data according to the spec in the top of the file.
The categories are reported separately for the start- and end-loops.
The end loop breakdown is shown only for the reads that reached the end loop.""",
}
print(caption[adapter_version])
print("\n\n\n\n")

In [ ]:
if sr_hist_png is not None:
    ipy_display(Image(sr_hist_png, width=IMAGE_WIDTH))
    print("Overall strand ratio (sr) histogram across all reads.")
    print("\n\n\n\n")
if sr_by_et_png is not None:
    ipy_display(Image(sr_by_et_png, width=IMAGE_WIDTH))
    print(
        "Strand ratio (sr) faceted by end-tag (et), "
        "restricted to reads where the read end was reached (tm contains 'A')."
    )
    print("\n\n\n\n")

In [ ]:
if strand_ratio_category_concordance_png is not None:
    ipy_display(Image(strand_ratio_category_concordance_png, width=IMAGE_WIDTH))
    caption = {
        PpmseqAdapterVersions.LEGACY_V5.value: (
            "These plots show the concordance between the strand ratio categories of the start-loop and end-loop. "
            "Each loop is assigned a category separately, and the concordance is plotted. The top plot includes all "
            "the reads, including those with END_UNREACHED, while the bottom includes reads where the end was reached "
            "only."
        ),
        PpmseqAdapterVersions.V1.value: (
            "These plots show the concordance between the strand ratio categories of the start-loop and end-loop. "
            "Each loop is assigned a category separately, and the concordance is plotted. The top plot includes all "
            "the reads, including those with END_UNREACHED, while the bottom includes reads where the end was reached "
            "only."
        ),
    }
    print(caption[adapter_version])
print("\n\n\n\n")

<\!-- Removed: Trimmer histogram plot is no longer generated since QC consumes the subsampled SAM directly. -->

In [ ]:
if output_read_length_histogram_plot is not None:
    ipy_display(Image(output_read_length_histogram_plot, width=IMAGE_WIDTH))

# About ppmSeq

Identifying single nucleotide variants (SNVs) is fundamental to genomics. While consensus mutation calling, requiring multiple variant-containing reads to call genetic variation, is often used, it is unsuitable in calling rare SNVs, such as in circulating tumor DNA or somatic mosaicism, where often only a single supporting read is available. Paired Plus and Minus strand Sequencing (ppmSeq), a PCR-free library preparation technology that uniquely leverages the Ultima Genomics clonal amplification process, overcomes this challenge. Here, DNA denaturation is not required prior to clonal amplification so both native strands are clonally amplified on many sequencing beads, allowing for a linear increase in duplex recovery and scalable duplex coverage without requiring unique molecular identifiers or redundant sequencing. 

In ppmSeq, modified Ultima Genomics adapters containing mismatched homopolymers are used to detect reads that are the result of the mixture of the two native DNA strands. While some reads are amplicons of only the Plus or Minus strands and are generally of typical UG read SNV accuracy, the so-called Mixed reads exhibit much lower error rates, well below 1E-6, facilitating the accurate detection of rare SNVs. Artifactual mutations manifesting on one strand only are common sources of error in SNV detection from NGS. While beads that are amplicons of Plus or Minus strand only are exposed to these artifacts that would appear as high-quality reads, in Mixed beads they create an inconsistent signal that translates into a low quality base or read, preventing them from being read as false positive SNVs. 

This report is generated from preprocessing of the ppmSeq sequencing data, and is intended to be used as a QC report for the library prep and sequencing run. The distribution of the MINUS/PLUS ratio, assignment of reads to categories (MIXED/MINUS/PLUS/UNDETERMINED), and with the raw calls are shown.

In [ ]:
if illustration_file is not None and os.path.isfile(illustration_file):
    ipy_display(Image(illustration_file, width=IMAGE_WIDTH))

# ppmSeq adapter version

In [ ]:
adapter_version_desc = {
    PpmseqAdapterVersions.LEGACY_V5_START.value: (
        "The ppmSeq_legacy_v5 (start-only) adapter is used in this sample. "
        "QC is derived from per-read sr/st/et/tm tags in the subsampled SAM."
    ),
    PpmseqAdapterVersions.LEGACY_V5_END.value: (
        "The ppmSeq_legacy_v5 (end-only) adapter is used in this sample. "
        "QC is derived from per-read sr/st/et/tm tags in the subsampled SAM."
    ),
    PpmseqAdapterVersions.LEGACY_V5.value: (
        "The ppmSeq_legacy_v5 adapter is used in this sample. "
        "QC is derived from per-read sr/st/et/tm tags in the subsampled SAM."
    ),
    PpmseqAdapterVersions.V1.value: (
        "The ppmSeq_v1 adapter is used in this sample. "
        "The st / et category tags are assigned either from base-calling-derived loop tokens "
        "(Solaris-1) or from the per-read sr (strand ratio) tag via a --compare cascade (Solaris-2)."
    ),
}
print(adapter_version_desc.get(adapter_version, f"Adapter version: {adapter_version}"))
print("\n\n\n\n")

# Detailed statistics

In [ ]:
with pd.HDFStore(statistics_h5) as s:
    keys = s.keys()
for key in sorted(keys):
    if key in ["/sorter_stats_shortlist", "sorter_stats_shortlist"]:  # already shown
        continue
    if key in ["/subsampled_reads", "subsampled_reads"]:  # too large, skip from report
        continue
    if key in ["/keys_to_convert", "keys_to_convert"]:  # metadata only
        continue
    print(f"\n\nStatistics table:{key.replace('/', ' ')}")
    df = pd.read_hdf(statistics_h5, key=key)  # noqa: PD901
    ipy_display(df)